# 02 — Train a DQN Model Offline

Load the transitions collected by `01_collect_dataset.ipynb` and train a small DQN-style MOUSE model entirely from stored data — no live environment needed.

The model has three independently-configurable pieces:

| Piece | Role |
|---|---|
| `StepEmbedder` | Encodes each `(action, observation, reward, done)` step into a sequence of vectors |
| `Qwen3Backbone` | A small transformer that reads those vectors and builds up context over time |
| `DiscreteActionValueHead` | Predicts a Q-value for each possible action |

`Qwen3Backbone` downloads a public pretrained checkpoint on first run and exposes `hidden_dim`, which sizes everything else.

In [1]:
import torch

from mouse_core.data import DataLoader, load_stores_from_hub
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead

# Hub dataset repo written by 01_collect_dataset.ipynb.
DATASET_ID = "mouse-example-dataset"

# Hub model repo where the trained checkpoint is uploaded.
MODEL_ID = "mouse-example-model"

# FrozenLake has four discrete actions, matching the DQN head width.
MAX_ACTIONS = 4

# All slots use 4×4 maps (16 discrete states, 0–15).
MAX_OBS_DISCRETE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load data

`load_stores_from_hub` downloads each named environment config from the Hub. The `DataLoader` accepts the list of stores directly and mixes them during training — no manual merge needed.

In [2]:
stores = load_stores_from_hub(DATASET_ID, split="train")
for s in stores:
    print(s)

Datastore(name='proc_frozenlake_0', steps=1501)
Datastore(name='proc_frozenlake_1', steps=1501)
Datastore(name='proc_frozenlake_2', steps=1501)
Datastore(name='proc_frozenlake_3', steps=1501)


## DataLoader

`DataLoader` slices one or more `Datastore`s into fixed-length sequence batches. Multiple stores are mixed automatically. Call `loader.next_batch()` to get a `list[list[dict]]` of shape `[B][S]` — one inner list per sequence in the batch, one dict per step. `model.forward` accepts this directly and handles encoding internally.

In [3]:
loader = DataLoader(stores, sequence_length=16, batch_size=32, prefetch=4, num_workers=0)

## Build the model

`Qwen3Backbone` downloads `Qwen/Qwen3-0.6B` on first run and keeps only the first `num_layers` transformer layers. `StepEmbedder` embeds each step into tokens, the backbone mixes them over time, and `DiscreteActionValueHead` maps the result to per-action Q-values — all sized by `backbone.hidden_dim`.

In [4]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B", num_layers=2)

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {"name": "action",      "embed": "discrete",  "vocab_size": MAX_ACTIONS,      "std": 0.02, "tokens": 1},
        {"name": "observation", "embed": "discrete",  "vocab_size": MAX_OBS_DISCRETE, "std": 0.02, "tokens": 1},
        {"name": "reward",      "embed": "rff",       "std": 0.02, "in_min": 0.01, "in_max": 1.0, "tokens": 1},
        {"name": "done",        "embed": "discrete",  "vocab_size": 5,               "std": 0.02, "tokens": 1},
        {"name": "scratch",     "embed": "learnable", "tokens": 1},
    ],
    concat_modalities=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.01,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(16, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
      (scratch): LearnableEmbedder()
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-1): 2 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj): Linea

## Train

Each step:
1. Sample a random batch of transition sequences from the stores.
2. Run the model forward to get Q-value predictions from both the online and frozen target networks.
3. Compute the Bellman loss — `objective(objective_data, predictions)` returns `(loss, metrics)`.
4. Update the model with gradient descent.
5. Slowly copy online weights to the frozen target network (Polyak update) for stable TD targets.

### Episode vs task discount factors

MOUSE uses five `done` codes; `DqnObjective` maps each to its own bootstrap discount:

| done | Meaning | Discount |
|---|---|---|
| 0 | Running | `gamma` |
| 1 | Episode terminated (not last in task) | `gamma_episode_terminal` |
| 2 | Episode truncated (not last in task) | `gamma_episode_truncated` |
| 3 | **Task terminated** (last episode done) | `gamma_task_terminal` |
| 4 | **Task truncated** (last episode trunc.) | `gamma_task_truncated` |

Within a task the episodes share context (same map, same task), so we **do** bootstrap across episode boundaries (`gamma_episode_terminal = gamma_episode_truncated = 1.0`). A task boundary ends the context entirely — the next task's Q-values carry no meaningful signal — so we **suppress** bootstrapping there (`gamma_task_terminal = gamma_task_truncated = 0.0`).

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
objective = DqnObjective(
    gamma=1.0,
    gamma_episode_terminal=1.0,   # bootstrap across episode boundaries within a task
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,      # no bootstrap across task boundaries
    gamma_task_truncated=0.0,
    tau=0.005,
)


def train_step(batch):
    predictions, objective_data, _ = model(batch)
    loss, _ = objective(objective_data, predictions)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    model.polyak_update(action_value_tau=objective.tau)
    return loss.item()


TRAIN_STEPS = 2000
for step in range(TRAIN_STEPS):
    loss = train_step(loader.next_batch())
    if step % 100 == 0:
        print(f"step={step:4d}  loss={loss:.4f}")

loader.close()
print("Training finished.")

step=   0  loss=0.0687


step= 100  loss=0.0004


step= 200  loss=0.0006


step= 300  loss=0.0009


step= 400  loss=0.0002


...

step=1600  loss=0.0036


step=1700  loss=0.0030


step=1800  loss=0.0065


step=1900  loss=0.0021


Training finished.


## Push to the Hub

Upload the trained checkpoint. The next notebook loads it by the same `MODEL_ID` to run inference.

In [6]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")

Pushed to https://huggingface.co/micahr234/mouse-example-model
